In [2]:
pip install scikit-learn


In [3]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import numpy as np


In [4]:
 from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
train_df = pd.read_csv("/content/drive/MyDrive/train_preprocessed.csv")
test_df = pd.read_csv("/content/drive/MyDrive/test_preprocessed.csv")

# Example columns: 'text', 'label'
train_texts = train_df['cleaned'].astype(str).tolist()
test_texts = test_df['cleaned'].astype(str).tolist()

label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(train_df['Sentiment'])
test_labels = label_encoder.transform(test_df['Sentiment'])
num_classes = len(label_encoder.classes_)

In [6]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT2 has no pad_token

def encode_texts(texts, tokenizer, max_len=128):
    return tokenizer(texts, truncation=True, padding='max_length', max_length=max_len, return_tensors="pt")

train_encodings = encode_texts(train_texts, tokenizer)
test_encodings = encode_texts(test_texts, tokenizer)


In [7]:
class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}, torch.tensor(self.labels[idx])

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, train_labels)
test_dataset = TextDataset(test_encodings, test_labels)


In [8]:
class CausalTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, num_heads, num_layers, num_classes, max_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = nn.Embedding(max_len, emb_dim)  # absolute position embedding
        self.transformer_blocks = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=emb_dim, nhead=num_heads),
            num_layers=num_layers
        )
        self.fc = nn.Linear(emb_dim, num_classes)

    def forward(self, input_ids):
        positions = torch.arange(0, input_ids.size(1)).unsqueeze(0).to(input_ids.device)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        x = self.transformer_blocks(x)
        x = x.mean(dim=1)  # Global average pooling
        return self.fc(x)

model = CausalTransformerClassifier(
    vocab_size=len(tokenizer),
    emb_dim=128,
    num_heads=4,
    num_layers=2,
    num_classes=num_classes,
    max_len=128
)


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [10]:
for epoch in range(5):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader):
        inputs, labels = batch
        inputs = inputs['input_ids'].to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")


100%|██████████| 9873/9873 [02:47<00:00, 59.11it/s]


Epoch 1 Loss: 7610.1854


100%|██████████| 9873/9873 [02:49<00:00, 58.17it/s]


Epoch 2 Loss: 6213.3287


100%|██████████| 9873/9873 [02:50<00:00, 57.97it/s]


Epoch 3 Loss: 5358.8021


100%|██████████| 9873/9873 [02:50<00:00, 57.98it/s]


Epoch 4 Loss: 4904.5391


100%|██████████| 9873/9873 [02:50<00:00, 57.86it/s]

Epoch 5 Loss: 4596.5250


In [11]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        inputs, labels = batch
        inputs = inputs['input_ids'].to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_))


              precision    recall  f1-score   support

    Negative       0.70      0.69      0.70     22984
     Neutral       0.74      0.52      0.61     23105
    Positive       0.87      0.95      0.91     89311

    accuracy                           0.83    135400
   macro avg       0.77      0.72      0.74    135400
weighted avg       0.82      0.83      0.82    135400



In [13]:
torch.save(model.state_dict(), "causal_transformer_a.pt")
